In [1]:
from datetime import datetime, timezone
import sqlite3
import pandas as pd
import requests

BINANCE_KLINES_URL = "https://api.binance.com/api/v3/klines"
DATABASE = "crypto_historical_data.db"

In [65]:
def get_latest_db_timestamp():
    """Fetch the max timestamp (in milliseconds) currently stored in the DB."""
    with sqlite3.connect(DATABASE) as conn:
        result = pd.read_sql("SELECT MAX(time_close) FROM btc_price", conn)
    result = result.max().values[0]
    dt = datetime.fromisoformat(result)
    ms = int(dt.timestamp() * 1000)
    return ms


def fetch_binance_klines(symbol="BTCUSDT", interval="1d", start_time=None):
    """Fetch candlestick data directly from Binance public REST API."""
    params = {
        "symbol": symbol,
        "interval": interval,
        "limit": 1000,  # Max allowed per request
    }
    if start_time:
        # +1 ms so we don't duplicate the last existing candle
        params["startTime"] = start_time + 1

    response = requests.get(BINANCE_KLINES_URL, params=params)
    response.raise_for_status()
    data = response.json()

    if not data:
        return pd.DataFrame()

    # Parse Binance KLine Array Structure
    cols = [
        "timestamp",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "quote_asset_volume",
        "number_of_trades",
        "taker_buy_base_asset_volume",
        "taker_buy_quote_asset_volume",
        "ignore",
    ]

    df = pd.DataFrame(data, columns=cols)

    # Clean & Format Columns
    df["symbol"] = symbol
    df = df[["timestamp", "symbol", "open", "high", "low", "close", "volume"]]
    numeric_cols = ["open", "high", "low", "close", "volume"]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

    return df

def update_btc_database(db_path=DATABASE):
    """Main function to perform delta update."""
    with sqlite3.connect(db_path) as conn:
        last_ts = get_latest_db_timestamp()
        new_candles = fetch_binance_klines(
            symbol="BTCUSDT", interval="1d", start_time=last_ts
        )

        if new_candles.empty:
            return pd.DataFrame()

        # Insert new rows into database
        new_candles = new_candles.rename(columns={"timestamp": "time_close"})
        new_candles['time_close'] = pd.to_datetime(new_candles['time_close'], unit='ms', utc=True)
        new_candles = new_candles[["time_close", "close", "high", "low", "open"]]
        new_candles.to_sql("btc_price", conn, if_exists="append", index=False)
    
    return new_candles


In [66]:
df = update_btc_database()

In [67]:
df

,time_close,close,high,low,open
0,2026-07-28 00:00:00+00:00,63915.00,64100.00,62742.47,63755.86
1,2026-07-29 00:00:00+00:00,63984.28,64744.81,63267.34,63915.00
2,2026-07-30 00:00:00+00:00,64780.02,65176.60,63603.92,63984.29
3,2026-07-31 00:00:00+00:00,62887.88,65409.56,62466.00,64780.03
4,2026-08-01 00:00:00+00:00,62823.64,63150.00,62275.00,62887.88
5,2026-08-02 00:00:00+00:00,63570.00,63796.33,62806.58,62823.65
6,2026-08-03 00:00:00+00:00,63520.00,64080.00,62300.00,63570.01
7,2026-08-04 00:00:00+00:00,64106.56,64549.16,63322.01,63520.00
8,2026-08-05 00:00:00+00:00,64665.23,65025.22,63880.00,64106.55
9,2026-08-06 00:00:00+00:00,64323.61,64999.00,64172.00,64665.24


In [3]:
with sqlite3.connect(DATABASE) as conn:
        df = pd.read_sql("SELECT time_close, close, high, low FROM btc_price", conn)
df

,time_close,close,high,low
0,2010-07-17T00:00:00+00:00,0.04951,0.04951,0.04951
1,2010-07-19T00:00:00+00:00,0.08080,0.09307,0.07723
2,2010-07-20T00:00:00+00:00,0.07474,0.07474,0.07426
3,2010-07-21T00:00:00+00:00,0.07921,0.07921,0.07171
4,2010-07-22T00:00:00+00:00,0.05248,0.05941,0.05050
...,...,...,...,...
5854,2026-08-30T00:00:00+00:00,78860.35000,79400.00000,77962.49000
5855,2026-08-31 00:00:00,78562.74000,79257.14000,77369.59000
5856,2026-09-01 00:00:00,77398.69000,79195.32000,76366.12000
5857,2026-09-02 00:00:00,77307.36000,77750.00000,76219.18000


In [64]:
df["time_close"]

0       2010-07-17T00:00:00+00:00
1       2010-07-19T00:00:00+00:00
2       2010-07-20T00:00:00+00:00
3       2010-07-21T00:00:00+00:00
4       2010-07-22T00:00:00+00:00
                  ...            
5850    2026-08-26T00:00:00+00:00
5851    2026-08-27T00:00:00+00:00
5852    2026-08-28T00:00:00+00:00
5853    2026-08-29T00:00:00+00:00
5854    2026-08-30T00:00:00+00:00
Name: time_close, Length: 5855, dtype: object